# LeetCode 1429 — First Unique Number bằng Python

Link đề: https://leetcode.com/problems/first-unique-number/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Design, Queue, Data Stream

> **Ghi chú Premium:** đây là bài **LeetCode Premium** (🔒). Đề bài và chữ ký lớp trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1429/). Nếu có tài khoản Premium, nên đối chiếu lại với đề gốc.
>
> Đây là bài **thiết kế lớp**: notebook tự dựng **bộ chạy chuỗi lệnh giả lập** kiểu LeetCode (`["FirstUnique","showFirstUnique",...]`) để kiểm thử — đó là bản giả lập, không phải trình chấm thật.

Skeleton LeetCode (theo doocs):

```python
class FirstUnique:
    def __init__(self, nums: list[int]):
        ...

    def showFirstUnique(self) -> int:
        ...

    def add(self, value: int) -> None:
        ...

# Your FirstUnique object will be instantiated and called as such:
# obj = FirstUnique(nums)
# param_1 = obj.showFirstUnique()
# obj.add(value)
```

## 1. Mục tiêu học

1. Thiết kế cấu trúc dữ liệu cho **luồng dữ liệu (data stream)**: vừa thêm phần tử, vừa truy vấn nhanh "phần tử **duy nhất** xuất hiện **sớm nhất**".
2. Kết hợp **bảng băm đếm** (`Counter`) với **cấu trúc giữ thứ tự** (`deque`, `OrderedDict`, `dict` thường từ Python 3.7, danh sách liên kết đôi).
3. Hiểu kỹ thuật **xoá lười (lazy deletion)** và phân tích **khấu hao (amortized)** `O(1)`.
4. Tự cài **danh sách liên kết đôi + hashmap** (bản chất của `LinkedHashSet` / `OrderedDict`) để xoá một phần tử ở giữa trong `O(1)`.
5. Tránh lỗi: dùng `set` (không giữ thứ tự), bật/tắt theo chẵn lẻ, `if` thay cho `while` khi xoá lười, quên đếm trùng ngay trong `nums` ban đầu.

## 2. Đề bài & ví dụ

Có một **hàng đợi** các số nguyên. Cần lấy ra **số duy nhất đầu tiên** (số chỉ xuất hiện đúng 1 lần trong hàng đợi, đứng trước nhất).

Cài đặt lớp `FirstUnique`:
- `FirstUnique(int[] nums)` — khởi tạo với các số trong hàng đợi.
- `int showFirstUnique()` — trả về giá trị số duy nhất đầu tiên của hàng đợi, hoặc `-1` nếu không có.
- `void add(int value)` — chèn `value` vào cuối hàng đợi.

**Ví dụ 1:**
```
["FirstUnique","showFirstUnique","add","showFirstUnique","add","showFirstUnique","add","showFirstUnique"]
[[[2,3,5]],[],[5],[],[2],[],[3],[]]
→ [null,2,null,2,null,3,null,-1]
```
Hàng đợi `[2,3,5]` → 2; thêm 5 `[2,3,5,5]` → 2; thêm 2 `[2,3,5,5,2]` → 3; thêm 3 → `-1`.

**Ví dụ 2:**
```
["FirstUnique","showFirstUnique","add","add","add","add","add","showFirstUnique"]
[[[7,7,7,7,7,7]],[],[7],[3],[3],[7],[17],[]]
→ [null,-1,null,null,null,null,null,17]
```

**Ví dụ 3:**
```
["FirstUnique","showFirstUnique","add","showFirstUnique"]
[[[809]],[],[809],[]]
→ [null,809,null,-1]
```

**Ràng buộc:** `1 <= nums.length <= 10^5`, `1 <= nums[i] <= 10^8`, `1 <= value <= 10^8`, tối đa `50000` lời gọi `showFirstUnique` và `add`.

In [1]:
import random
import time
from collections import Counter, OrderedDict, deque
from typing import List

def run_ops(cls, ops, args):
    """BỘ CHẠY GIẢ LẬP kiểu LeetCode: ops[0] là tên lớp, args[0] = [nums]."""
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == "FirstUnique":
            obj = cls(*a)
            out.append(None)
        elif op == "showFirstUnique":
            out.append(obj.showFirstUnique())
        elif op == "add":
            out.append(obj.add(*a))
        else:
            raise ValueError(op)
    return out

examples = [
    (["FirstUnique","showFirstUnique","add","showFirstUnique","add","showFirstUnique","add","showFirstUnique"],
     [[[2,3,5]],[],[5],[],[2],[],[3],[]], [None,2,None,2,None,3,None,-1]),
    (["FirstUnique","showFirstUnique","add","add","add","add","add","showFirstUnique"],
     [[[7,7,7,7,7,7]],[],[7],[3],[3],[7],[17],[]], [None,-1,None,None,None,None,None,17]),
    (["FirstUnique","showFirstUnique","add","showFirstUnique"], [[[809]],[],[809],[]], [None,809,None,-1]),
]
for ops, args, want in examples:
    print(ops[1:], "->", want)

['showFirstUnique', 'add', 'showFirstUnique', 'add', 'showFirstUnique', 'add', 'showFirstUnique'] -> [None, 2, None, 2, None, 3, None, -1]
['showFirstUnique', 'add', 'add', 'add', 'add', 'add', 'showFirstUnique'] -> [None, -1, None, None, None, None, None, 17]
['showFirstUnique', 'add', 'showFirstUnique'] -> [None, 809, None, -1]


## 3. Kiến thức nền

### 3.1 `Counter` — đếm tần suất

`Counter(nums)` đếm số lần mỗi giá trị; `cnt[x] += 1` khi thêm. Khoá vắng mặt trả `0`.

In [2]:
cnt = Counter([2, 3, 5])
cnt[5] += 1
print(cnt, cnt[2], cnt[99])

Counter({5: 2, 2: 1, 3: 1}) 1 0


### 3.2 `deque` — hàng đợi hai đầu

`append` (cuối), `popleft` (đầu), xem đầu bằng `q[0]` — đều `O(1)`. Khác với `list.pop(0)` là `O(n)`.

In [3]:
q = deque([2, 3, 5])
q.append(5)
print(q[0], q.popleft(), list(q))

2 2 [3, 5, 5]


### 3.3 `dict` giữ thứ tự chèn (Python 3.7+) và `OrderedDict`

Từ Python 3.7, `dict` **giữ thứ tự chèn**; `next(iter(d))` lấy khoá chèn sớm nhất còn lại, `del d[k]` / `d.pop(k)` xoá `O(1)`. `OrderedDict` có thêm `move_to_end`, `popitem(last=False)`. Dùng như một **"tập có thứ tự"**.

In [4]:
d = {}
for x in [2, 3, 5]:
    d[x] = None
del d[2]
print(list(d), "đầu tiên:", next(iter(d)))
od = OrderedDict.fromkeys([4, 1, 9])
od.pop(1)
print(list(od), next(iter(od)))

[3, 5] đầu tiên: 3
[4, 9] 4


### 3.4 `set` **không** giữ thứ tự

`set` xoá/thêm `O(1)` nhưng thứ tự duyệt phụ thuộc giá trị băm, **không** phải thứ tự chèn ⇒ không dùng để lấy "phần tử đầu tiên".

In [5]:
s = set()
for x in [5, 1, 100, 3]:
    s.add(x)
print("chèn 5,1,100,3 -> duyệt set:", list(s))

chèn 5,1,100,3 -> duyệt set: [1, 3, 100, 5]


### 3.5 Danh sách liên kết đôi (doubly linked list)

Mỗi nút có `prev`, `next`. Có tham chiếu tới nút ⇒ gỡ nút khỏi danh sách trong `O(1)`. Dùng **nút gác (sentinel)** `head`, `tail` để khỏi xử lý trường hợp rỗng.

In [6]:
class Node:
    __slots__ = ("val", "prev", "next")
    def __init__(self, val=0):
        self.val, self.prev, self.next = val, None, None

head, tail = Node(), Node()
head.next, tail.prev = tail, head
def push_back(node):
    node.prev, node.next = tail.prev, tail
    tail.prev.next = node; tail.prev = node
def unlink(node):
    node.prev.next, node.next.prev = node.next, node.prev

a, b, c = Node(1), Node(2), Node(3)
for nd in (a, b, c): push_back(nd)
unlink(b)
cur, vals = head.next, []
while cur is not tail:
    vals.append(cur.val); cur = cur.next
print(vals)

[1, 3]


## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1, theo dõi **hàng đợi**, **số lần xuất hiện** và **các số đang duy nhất theo thứ tự**:

| thao tác | hàng đợi | đếm | duy nhất (theo thứ tự) | show |
|---|---|---|---|---|
| init `[2,3,5]` | 2 3 5 | 2:1 3:1 5:1 | 2, 3, 5 | |
| show | | | | **2** |
| add 5 | 2 3 5 5 | 5:2 | 2, 3 | |
| show | | | | **2** |
| add 2 | 2 3 5 5 2 | 2:2 | 3 | |
| show | | | | **3** |
| add 3 | … 3 | 3:2 | (rỗng) | |
| show | | | | **−1** |

Nhận xét:
- Một số **chỉ duy nhất khi đếm = 1**. Khi đếm lên 2 nó **mất tư cách vĩnh viễn** (đếm không bao giờ giảm).
- "Đầu tiên" theo thứ tự **lần xuất hiện đầu tiên** của số đó trong hàng đợi.
- ⇒ Cần: đếm (`Counter`) + danh sách các số duy nhất **giữ thứ tự**, xoá được nhanh khi một số trở thành trùng.

In [7]:
queue, cnt = [2, 3, 5], Counter([2, 3, 5])
for v in [None, 5, None, 2, None, 3, None]:
    if v is None:
        first = next((x for x in queue if cnt[x] == 1), -1)
        print("show ->", first, "| hàng đợi", queue)
    else:
        queue.append(v); cnt[v] += 1

show -> 2 | hàng đợi [2, 3, 5]
show -> 2 | hàng đợi [2, 3, 5, 5]
show -> 3 | hàng đợi [2, 3, 5, 5, 2]
show -> -1 | hàng đợi [2, 3, 5, 5, 2, 3]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lưu nguyên hàng đợi (list) và `Counter`. Mỗi lần `showFirstUnique` duyệt hàng đợi từ đầu, trả số đầu tiên có đếm bằng 1.

In [8]:
class FirstUniqueBrute:
    def __init__(self, nums: list[int]):
        self.queue = list(nums)
        self.cnt = Counter(nums)

    def showFirstUnique(self) -> int:
        for x in self.queue:                 # quét từ đầu mỗi lần
            if self.cnt[x] == 1:
                return x
        return -1

    def add(self, value: int) -> None:
        self.queue.append(value)
        self.cnt[value] += 1

for ops, args, want in examples:
    got = run_ops(FirstUniqueBrute, ops, args)
    print(got, "✔" if got == want else "✘")

[None, 2, None, 2, None, 3, None, -1] ✔
[None, -1, None, None, None, None, None, 17] ✔
[None, 809, None, -1] ✔


**Độ phức tạp:** `add` `O(1)`; `showFirstUnique` `O(L)` với `L` = độ dài hàng đợi (tới `1.5·10^5`). Với `5·10^4` lần `show` khi phần đầu hàng đợi toàn số trùng ⇒ `~7.5·10^9` bước ⇒ TLE.

(Bản "ngây thơ" hơn nữa dùng `queue.count(x)` thay cho `Counter` ⇒ `O(L²)` mỗi lần show.)

**Nút thắt:** mỗi lần show lại duyệt qua **cùng** những số trùng ở đầu hàng đợi. Số đã trùng thì **trùng mãi mãi** ⇒ có thể **vứt bỏ hẳn** chúng.

In [9]:
n = 20000
nums = [i // 2 for i in range(n)] + [10**8]           # nửa đầu toàn cặp trùng, số duy nhất ở cuối
fu = FirstUniqueBrute(nums)
t = time.perf_counter()
for _ in range(200):
    fu.showFirstUnique()
print(f"brute: 200 lần show trên hàng đợi {len(nums)} phần tử: {time.perf_counter() - t:.3f}s "
      f"(5·10^4 lần show sẽ lâu hơn ~250 lần)")

brute: 200 lần show trên hàng đợi 20001 phần tử: 0.137s (5·10^4 lần show sẽ lâu hơn ~250 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — trùng là vĩnh viễn:** đếm chỉ tăng ⇒ khi `cnt[x] ≥ 2`, `x` không bao giờ là đáp án nữa ⇒ **xoá được** khỏi mọi cấu trúc ứng viên.

**Insight 2 — xoá lười ở đầu hàng đợi:** giữ `deque` theo thứ tự xuất hiện. Khi `show`: **trong khi** phần tử đầu có `cnt ≥ 2` **thì** `popleft`. Mỗi phần tử bị pop **tối đa một lần** ⇒ tổng chi phí pop trên mọi lần gọi ≤ tổng số phần tử ⇒ **khấu hao `O(1)`**.

**Insight 3 — xoá ngay (eager) bằng tập có thứ tự:** giữ `unique` = các số có `cnt == 1`, theo thứ tự chèn (`OrderedDict`/`dict`/linked list).
- `add(v)`: `cnt[v] += 1`; **nếu** `cnt[v] == 1` **thì** chèn `v` vào cuối `unique`; **nếu** `cnt[v] == 2` **thì** xoá `v` khỏi `unique`; lớn hơn thì không làm gì.
- `show`: phần tử đầu của `unique` hoặc `-1` — `O(1)` thật sự (không khấu hao).

**Bất biến:** `unique` chứa đúng các giá trị có `cnt == 1`, theo thứ tự lần xuất hiện đầu tiên.

## 7. Các cách tiếp theo

### Cách 2 — `deque` + `Counter`, xoá lười

**Từng đoạn:**
1. `__init__`: `cnt = Counter(nums)`, `q = deque(nums)`.
2. `add`: `cnt[v] += 1`, `q.append(v)` (có thể thêm điều kiện chỉ append khi `cnt[v] == 1` để hàng đợi gọn hơn).
3. `show`: `while q and cnt[q[0]] > 1: q.popleft()`; trả `q[0]` hoặc `-1`.

In [10]:
class FirstUniqueLazy:
    def __init__(self, nums: list[int]):
        self.cnt = Counter(nums)
        self.q = deque(nums)

    def showFirstUnique(self) -> int:
        while self.q and self.cnt[self.q[0]] > 1:
            self.q.popleft()                 # vứt số đã trùng ở đầu
        return self.q[0] if self.q else -1

    def add(self, value: int) -> None:
        self.cnt[value] += 1
        if self.cnt[value] == 1:             # chỉ số mới có thể là ứng viên
            self.q.append(value)

for ops, args, want in examples:
    got = run_ops(FirstUniqueLazy, ops, args)
    print(got, got == want)

[None, 2, None, 2, None, 3, None, -1] True
[None, -1, None, None, None, None, None, 17] True
[None, 809, None, -1] True


**Độ phức tạp:** `__init__` `O(n)`; `add` `O(1)`; `show` khấu hao `O(1)` (mỗi phần tử popleft tối đa 1 lần). Bộ nhớ `O(n + số lần add)`.

### Cách 3 — `Counter` + `OrderedDict` (hoặc `dict`) làm "tập có thứ tự" (theo doocs)

**Từng đoạn:**
1. `__init__`: `cnt = Counter(nums)`; `unique = OrderedDict` các `v` có `cnt[v] == 1`, theo thứ tự trong `nums`.
2. `show`: `next(iter(unique))` nếu không rỗng, ngược lại `-1`.
3. `add`: tăng đếm; `== 1` ⇒ chèn; còn nếu `v` đang ở `unique` ⇒ xoá.

In [11]:
class FirstUniqueOD:
    def __init__(self, nums: list[int]):
        self.cnt = Counter(nums)
        self.unique = OrderedDict((v, 1) for v in nums if self.cnt[v] == 1)

    def showFirstUnique(self) -> int:
        return -1 if not self.unique else next(iter(self.unique))

    def add(self, value: int) -> None:
        self.cnt[value] += 1
        if self.cnt[value] == 1:
            self.unique[value] = 1
        elif value in self.unique:
            self.unique.pop(value)

for ops, args, want in examples:
    got = run_ops(FirstUniqueOD, ops, args)
    print(got, got == want)

[None, 2, None, 2, None, 3, None, -1] True
[None, -1, None, None, None, None, None, 17] True
[None, 809, None, -1] True


**Độ phức tạp:** mọi thao tác `O(1)` (không khấu hao), `__init__` `O(n)`. Từ Python 3.7 có thể thay `OrderedDict` bằng `dict` thường; nhưng chú ý: `next(iter(d))` trên `dict` đã bị xoá nhiều phần tử ở đầu có thể tốn thời gian bỏ qua các ô trống trong bảng băm nội bộ (CPython gọn lại khi resize), nên `OrderedDict` (danh sách liên kết bên trong) an toàn hơn về lý thuyết.

### Cách 4 — Tự cài danh sách liên kết đôi + hashmap

Đây là "ruột" của `OrderedDict` / `LinkedHashSet` trong Java. Dùng khi phỏng vấn yêu cầu không dùng thư viện.

**Từng đoạn:**
1. `cnt` (dict đếm), `node_of` (giá trị → nút trong danh sách), sentinel `head`/`tail`.
2. `add(v)`: tăng đếm; `== 1` ⇒ tạo nút, nối vào cuối, lưu `node_of[v]`; `== 2` ⇒ gỡ nút `node_of[v]`, xoá khỏi map.
3. `show`: `head.next` (nếu khác `tail`).

In [12]:
class FirstUniqueLinked:
    class _Node:
        __slots__ = ("val", "prev", "next")
        def __init__(self, val=0):
            self.val, self.prev, self.next = val, None, None

    def __init__(self, nums: list[int]):
        self.head, self.tail = self._Node(), self._Node()
        self.head.next, self.tail.prev = self.tail, self.head
        self.cnt = {}
        self.node_of = {}
        for v in nums:
            self.add(v)

    def _push_back(self, node):
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def _unlink(self, node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def showFirstUnique(self) -> int:
        return -1 if self.head.next is self.tail else self.head.next.val

    def add(self, value: int) -> None:
        c = self.cnt.get(value, 0) + 1
        self.cnt[value] = c
        if c == 1:
            node = self._Node(value)
            self._push_back(node)
            self.node_of[value] = node
        elif c == 2:
            self._unlink(self.node_of.pop(value))

for ops, args, want in examples:
    got = run_ops(FirstUniqueLinked, ops, args)
    print(got, got == want)

[None, 2, None, 2, None, 3, None, -1] True
[None, -1, None, None, None, None, None, 17] True
[None, 809, None, -1] True


**Độ phức tạp:** mọi thao tác `O(1)`, bộ nhớ `O(số giá trị khác nhau)`.

## 8. TRACE từng bước

Trace cách 2 (xoá lười) trên ví dụ 2:

| thao tác | `cnt` (rút gọn) | `q` sau thao tác | trả về |
|---|---|---|---|
| init `[7×6]` | 7:6 | 7 7 7 7 7 7 | |
| show | | (pop hết 7) rỗng | −1 |
| add 7 | 7:7 | rỗng (không append vì cnt≠1) | |
| add 3 | 3:1 | 3 | |
| add 3 | 3:2 | 3 | |
| add 7 | 7:8 | 3 | |
| add 17 | 17:1 | 3 17 | |
| show | | pop 3 ⇒ 17 | 17 |

In [13]:
def trace_lazy(ops, args):
    obj = None
    for op, a in zip(ops, args):
        if op == "FirstUnique":
            obj = FirstUniqueLazy(*a)
            print(f"init {a[0]}: q={list(obj.q)}, cnt={dict(obj.cnt)}")
        elif op == "add":
            obj.add(*a)
            print(f"add {a[0]:>3}: q={list(obj.q)}, cnt={dict(obj.cnt)}")
        else:
            before = list(obj.q)
            r = obj.showFirstUnique()
            popped = len(before) - len(obj.q)
            print(f"show    : pop {popped} phần tử trùng ở đầu -> q={list(obj.q)} => trả {r}")

trace_lazy(*examples[1][:2])

init [7, 7, 7, 7, 7, 7]: q=[7, 7, 7, 7, 7, 7], cnt={7: 6}
show    : pop 6 phần tử trùng ở đầu -> q=[] => trả -1
add   7: q=[], cnt={7: 7}
add   3: q=[3], cnt={7: 7, 3: 1}
add   3: q=[3], cnt={7: 7, 3: 2}
add   7: q=[3], cnt={7: 8, 3: 2}
add  17: q=[3, 17], cnt={7: 8, 3: 2, 17: 1}
show    : pop 1 phần tử trùng ở đầu -> q=[17] => trả 17


Trace cách 3 (`OrderedDict`) trên ví dụ 1 — in tập duy nhất sau mỗi thao tác:

In [14]:
def trace_od(ops, args):
    obj = None
    for op, a in zip(ops, args):
        if op == "FirstUnique":
            obj = FirstUniqueOD(*a); msg = f"init {a[0]}"
        elif op == "add":
            obj.add(*a); msg = f"add {a[0]}"
        else:
            msg = f"show -> {obj.showFirstUnique()}"
        print(f"{msg:14s} | unique={list(obj.unique)} | cnt={dict(obj.cnt)}")

trace_od(*examples[0][:2])

init [2, 3, 5] | unique=[2, 3, 5] | cnt={2: 1, 3: 1, 5: 1}
show -> 2      | unique=[2, 3, 5] | cnt={2: 1, 3: 1, 5: 1}
add 5          | unique=[2, 3] | cnt={2: 1, 3: 1, 5: 2}
show -> 2      | unique=[2, 3] | cnt={2: 1, 3: 1, 5: 2}
add 2          | unique=[3] | cnt={2: 2, 3: 1, 5: 2}
show -> 3      | unique=[3] | cnt={2: 2, 3: 1, 5: 2}
add 3          | unique=[] | cnt={2: 2, 3: 2, 5: 2}
show -> -1     | unique=[] | cnt={2: 2, 3: 2, 5: 2}


## 9. Code nộp LeetCode

Dùng cách 3 (`Counter` + `OrderedDict`) — đúng tên lớp và phương thức theo doocs, mọi thao tác `O(1)`:

In [15]:
class FirstUnique:
    def __init__(self, nums: list[int]):
        self.cnt = Counter(nums)
        # các số đang duy nhất, theo thứ tự xuất hiện
        self.unique = OrderedDict((v, 1) for v in nums if self.cnt[v] == 1)

    def showFirstUnique(self) -> int:
        return -1 if not self.unique else next(iter(self.unique))

    def add(self, value: int) -> None:
        self.cnt[value] += 1
        if self.cnt[value] == 1:
            self.unique[value] = 1          # số mới -> ứng viên cuối hàng
        elif value in self.unique:
            self.unique.pop(value)          # vừa thành trùng -> loại vĩnh viễn

# Your FirstUnique object will be instantiated and called as such:
# obj = FirstUnique(nums)
# param_1 = obj.showFirstUnique()
# obj.add(value)

for ops, args, want in examples:
    assert run_ops(FirstUnique, ops, args) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

**Bất biến (cách 3/4):** sau mỗi thao tác, `unique` chứa **đúng** các giá trị `v` có `cnt[v] == 1`, sắp theo vị trí xuất hiện (duy nhất) của chúng trong hàng đợi.
- Khởi tạo: lọc `nums` theo `cnt == 1`, giữ thứ tự ⇒ đúng.
- `add(v)` với `cnt` mới = 1: `v` là số mới ở **cuối** hàng đợi ⇒ chèn cuối giữ thứ tự. `cnt` mới = 2: `v` không còn duy nhất ⇒ xoá. `cnt ≥ 3`: `v` đã bị xoá trước đó ⇒ không đổi. Các giá trị khác không đổi đếm.
- Vậy phần tử đầu của `unique` là số duy nhất có vị trí nhỏ nhất = đáp án.

**Cách 2 (lười):** bất biến: `q` chứa (theo thứ tự xuất hiện) mọi số có `cnt == 1`, có thể **lẫn thêm** số đã trùng. Số đã trùng không bao giờ trở lại duy nhất ⇒ pop chúng ở đầu là an toàn. Sau vòng `while`, đầu `q` (nếu có) có `cnt == 1` và mọi số duy nhất đều nằm sau hoặc tại nó ⇒ đúng.

**Khấu hao:** mỗi giá trị được `append` vào `q` tối đa 1 lần (chỉ khi `cnt` lên 1) và `popleft` tối đa 1 lần ⇒ tổng số pop ≤ số phần tử ⇒ `O(1)` trung bình mỗi lời gọi.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `m` = số lời gọi (`≤ 5·10^4`), `L = n + m`.

| Cách | `__init__` | `add` | `showFirstUnique` | Space |
|---|---|---|---|---|
| 1. Brute quét hàng đợi | `O(n)` | `O(1)` | `O(L)` | `O(L)` |
| 2. Deque + xoá lười | `O(n)` | `O(1)` | khấu hao `O(1)`, tệ nhất `O(L)` một lần | `O(L)` |
| 3. `OrderedDict` | `O(n)` | `O(1)` | `O(1)` | `O(số giá trị khác nhau)` |
| 4. Linked list + map | `O(n)` | `O(1)` | `O(1)` | `O(số giá trị khác nhau)` |

In [16]:
rng = random.Random(1429)
nums = [rng.randint(1, 50000) for _ in range(10**5)]
ops = [("add", rng.randint(1, 60000)) if rng.random() < 0.5 else ("show", None) for _ in range(50000)]
for cls in [FirstUniqueLazy, FirstUniqueOD, FirstUniqueLinked]:
    t = time.perf_counter()
    obj = cls(nums); acc = 0
    for op, v in ops:
        if op == "add": obj.add(v)
        else: acc += obj.showFirstUnique()
    print(f"{cls.__name__:18s}: {time.perf_counter() - t:.3f}s (checksum {acc})")

FirstUniqueLazy   : 0.022s (checksum 513839040)
FirstUniqueOD     : 0.037s (checksum 513839040)


FirstUniqueLinked : 0.060s (checksum 513839040)


## 12. Bộ test

1. Ba ví dụ đề (bằng bộ chạy chuỗi lệnh giả lập).
2. Edge: `nums` một phần tử; `nums` toàn trùng; thêm một số lần thứ 3 (không được "sống lại"); số mới thêm sau khi hết số duy nhất; số duy nhất đầu bị trùng ở giữa chuỗi lệnh.
3. Đối chiếu ngẫu nhiên các cách với brute force trên chuỗi lệnh dài (seed cố định).

In [17]:
impls = [FirstUniqueBrute, FirstUniqueLazy, FirstUniqueOD, FirstUniqueLinked, FirstUnique]
edge = [
    (["FirstUnique","showFirstUnique"], [[[1]],[]], [None, 1]),
    (["FirstUnique","showFirstUnique","add","add","showFirstUnique"], [[[4]],[],[4],[4],[]], [None,4,None,None,-1]),
    (["FirstUnique","add","showFirstUnique"], [[[9,9]],[5],[]], [None,None,5]),
    (["FirstUnique","showFirstUnique","add","showFirstUnique","add","showFirstUnique"],
     [[[1,2,3]],[],[1],[],[2],[]], [None,1,None,2,None,3]),
    (["FirstUnique","add","add","add","showFirstUnique"], [[[5,1]],[5],[5],[6],[]], [None,None,None,None,1]),
]
for cls in impls:
    for ops, args, want in examples + edge:
        assert run_ops(cls, ops, args) == want, (cls.__name__, ops, args)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [18]:
def random_ops(rng, n_ops, val_range):
    nums = [rng.randint(1, val_range) for _ in range(rng.randint(1, 8))]
    ops, args = ["FirstUnique"], [[nums]]
    for _ in range(n_ops):
        if rng.random() < 0.5:
            ops.append("add"); args.append([rng.randint(1, val_range)])
        else:
            ops.append("showFirstUnique"); args.append([])
    return ops, args

rng = random.Random(2020)
for _ in range(500):
    ops, args = random_ops(rng, rng.randint(1, 40), rng.choice([3, 6, 20]))
    ref = run_ops(FirstUniqueBrute, ops, args)
    for cls in impls[1:]:
        assert run_ops(cls, ops, args) == ref, (cls.__name__, ops, args)
print("random 500 chuỗi lệnh: OK")

random 500 chuỗi lệnh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `set` cho các số duy nhất (mất thứ tự)

In [19]:
class WrongSet:
    def __init__(self, nums):
        self.cnt = Counter(nums)
        self.unique = {v for v in nums if self.cnt[v] == 1}     # SAI: set không có thứ tự
    def showFirstUnique(self):
        return next(iter(self.unique)) if self.unique else -1
    def add(self, value):
        self.cnt[value] += 1
        if self.cnt[value] == 1: self.unique.add(value)
        else: self.unique.discard(value)

ops, args = ["FirstUnique", "showFirstUnique"], [[[100, 5, 1]], []]
print("sai:", run_ops(WrongSet, ops, args), "| đúng:", run_ops(FirstUnique, ops, args))

sai: [None, 1] | đúng: [None, 100]


**Vì sao sai:** thứ tự duyệt `set` của số nguyên nhỏ theo giá trị băm (thường gần như tăng dần), không theo thứ tự chèn ⇒ trả `1` thay vì `100`.

**Sửa:** dùng `OrderedDict`, `dict` (3.7+), `deque` hoặc danh sách liên kết — cấu trúc **giữ thứ tự chèn**.

### Lỗi 2 — Bật/tắt theo chẵn lẻ (số trùng "sống lại" ở lần thứ 3)

In [20]:
class WrongToggle:
    def __init__(self, nums):
        self.unique = OrderedDict()
        for v in nums: self.add(v)
    def showFirstUnique(self):
        return next(iter(self.unique)) if self.unique else -1
    def add(self, value):
        if value in self.unique: self.unique.pop(value)          # SAI: không nhớ đã từng trùng
        else: self.unique[value] = 1

ops, args = ["FirstUnique", "add", "add", "showFirstUnique"], [[[2]], [2], [2], []]
print("sai:", run_ops(WrongToggle, ops, args), "| đúng:", run_ops(FirstUnique, ops, args))

sai: [None, None, None, 2] | đúng: [None, None, None, -1]


**Vì sao sai:** sau `[2, 2, 2]`, số 2 xuất hiện 3 lần — không duy nhất. Code bật/tắt thấy 2 không có trong `unique` nên chèn lại ⇒ trả `2`.

**Sửa:** phải có **bộ đếm** (`Counter`): chỉ chèn khi đếm vừa lên 1, chỉ xoá khi vừa lên 2.

### Lỗi 3 — Xoá lười bằng `if` thay vì `while`

In [21]:
class WrongIf:
    def __init__(self, nums):
        self.cnt = Counter(nums); self.q = deque(nums)
    def showFirstUnique(self):
        if self.q and self.cnt[self.q[0]] > 1:                  # SAI: chỉ bỏ 1 phần tử
            self.q.popleft()
        return self.q[0] if self.q else -1
    def add(self, value):
        self.cnt[value] += 1; self.q.append(value)

ops, args = ["FirstUnique", "add", "add", "showFirstUnique"], [[[2, 3, 4]], [2], [3], []]
print("sai:", run_ops(WrongIf, ops, args), "| đúng:", run_ops(FirstUnique, ops, args))

sai: [None, None, None, 3] | đúng: [None, None, None, 4]


**Vì sao sai:** đầu hàng đợi có thể có **nhiều** số trùng liên tiếp (`2, 3` đều đã trùng). `if` chỉ bỏ `2`, rồi trả `3` — một số trùng.

**Sửa:** `while self.q and self.cnt[self.q[0]] > 1: self.q.popleft()`.

### Lỗi 4 — Không đếm trùng ngay trong `nums` ban đầu

In [22]:
class WrongInit:
    def __init__(self, nums):
        self.cnt = Counter()                                      # SAI: bỏ qua nums khi đếm
        self.unique = OrderedDict.fromkeys(nums)                  # và coi mọi số là duy nhất
    def showFirstUnique(self):
        return next(iter(self.unique)) if self.unique else -1
    def add(self, value):
        self.cnt[value] += 1
        if self.cnt[value] == 1: self.unique[value] = None
        elif value in self.unique: self.unique.pop(value)

ops, args = examples[1][:2]
print("sai:", run_ops(WrongInit, ops, args))
print("đúng:", run_ops(FirstUnique, ops, args))

sai: [None, 7, None, None, None, None, None, 17]
đúng: [None, -1, None, None, None, None, None, 17]


**Vì sao sai:** `nums = [7,7,7,7,7,7]` đã trùng ngay từ đầu, nhưng code coi `7` là duy nhất ⇒ show đầu tiên trả `7` thay vì `-1`; và vì `cnt` không biết về `nums`, `add(7)` lần đầu lại bị coi như số mới.

**Sửa:** khởi tạo `cnt = Counter(nums)` rồi lọc `cnt[v] == 1` (hoặc gọi `self.add(v)` cho từng `v` như cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Phần tử đầu tiên thoả tính chất X** trong luồng dữ liệu, và tính chất mất đi **vĩnh viễn**" ⇒ hàng đợi + **xoá lười** (khấu hao `O(1)`).
- Cần xoá phần tử **ở giữa** mà vẫn giữ thứ tự ⇒ **hashmap + danh sách liên kết đôi** (`OrderedDict`) — giống LRU Cache (146).
- Python: `dict` giữ thứ tự chèn từ 3.7; `next(iter(d))` lấy phần tử đầu; `OrderedDict.popitem(last=False)` lấy và xoá đầu.
- Đếm tăng đơn điệu ⇒ chỉ cần phản ứng ở **hai mốc**: lên 1 (chèn) và lên 2 (xoá).
- Test nhanh: số xuất hiện 3 lần, `nums` toàn trùng, nhiều số trùng liên tiếp ở đầu.

## 15. Biến thể của bài

1. **LeetCode 387 First Unique Character in a String**: phiên bản tĩnh — đếm rồi duyệt lại.
2. **First non-repeating character in a stream** (bài kinh điển GfG): y hệt nhưng trên ký tự.
3. **LRU Cache (146)**: cùng cấu trúc hashmap + danh sách liên kết đôi, thao tác "đưa về cuối".
4. **Hỗ trợ thêm `showLastUnique()`**: lấy phần tử cuối của tập có thứ tự (`next(reversed(od))`).
5. **Số duy nhất đầu tiên trong cửa sổ trượt `k` phần tử gần nhất** ⇒ đếm phải **giảm** khi phần tử rời cửa sổ ⇒ số có thể "sống lại", cần cấu trúc khác (ví dụ map vị trí + heap/SortedList).

Code biến thể 1:

In [23]:
def first_uniq_char(s):
    cnt = Counter(s)
    for i, ch in enumerate(s):
        if cnt[ch] == 1:
            return i
    return -1

print(first_uniq_char("leetcode"), first_uniq_char("loveleetcode"), first_uniq_char("aabb"))

0 2 -1


## 16. So sánh các cách

| Cách | Ý tưởng | `add` | `show` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Brute | Quét hàng đợi tìm số có đếm 1 | `O(1)` | `O(L)` | `O(L)` | Kiểm chứng |
| 2. Deque + lười | Pop số trùng ở đầu khi show | `O(1)` | khấu hao `O(1)` | `O(L)` | Đơn giản, nhanh |
| 3. `OrderedDict` | Tập có thứ tự các số đếm = 1 | `O(1)` | `O(1)` | `O(D)` | Mặc định trong Python (doocs) |
| 4. Linked list + map | Tự cài `LinkedHashSet` | `O(1)` | `O(1)` | `O(D)` | Phỏng vấn cấm thư viện |

(`L` = độ dài hàng đợi, `D` = số giá trị khác nhau.)

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
```
["FirstUnique","showFirstUnique","add","showFirstUnique","add","add","showFirstUnique"]
[[[1,1,2,3]],[],[2],[],[4],[3],[]]
```
Kết quả là gì?

### Bài 2 (tính tay)
Với cách 2 (deque lười), cho `nums = [5,6,5,7,6]`, rồi `show`, `add(7)`, `show`, `add(8)`, `show`: ghi `q` và `cnt` sau mỗi thao tác.

### Bài 3 (điền TODO)
Hoàn thiện lớp `FirstUniqueDict` dùng **`dict` thường** (không `OrderedDict`) làm tập có thứ tự.

### Bài 4 (tự viết lại)
Viết `first_non_repeating_stream(s)`: với chuỗi `s`, trả về chuỗi `out` cùng độ dài, `out[i]` là ký tự không lặp đầu tiên của tiền tố `s[:i+1]`, hoặc `'#'` nếu không có.

### Bài 5 (mini challenge)
Viết lớp `FirstLastUnique` có thêm `showLastUnique()` (số duy nhất **xuất hiện muộn nhất**) và `countUnique()` (số lượng số đang duy nhất), mọi thao tác `O(1)`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class FirstUniqueDict:
    def __init__(self, nums: list[int]):
        self.cnt = {}
        self.unique = {}                    # dict giữ thứ tự chèn
        # TODO: gọi self.add(v) cho từng v

    def showFirstUnique(self) -> int:
        # TODO: next(iter(self.unique)) hoặc -1
        pass

    def add(self, value: int) -> None:
        # TODO: tăng đếm; ==1 chèn; ==2 xoá
        pass

# Bài 4 — TODO
def first_non_repeating_stream(s):
    # TODO: Counter + deque, xoá lười ở đầu
    pass

# Bài 5 — TODO
class FirstLastUnique:
    def __init__(self, nums):
        # TODO: Counter + OrderedDict
        pass
    def showFirstUnique(self):
        pass
    def showLastUnique(self):
        # TODO: next(reversed(self.unique))
        pass
    def countUnique(self):
        pass
    def add(self, value):
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ban đầu `1` trùng, duy nhất: `2, 3` ⇒ show `2`. (2) `add(2)` ⇒ `2` trùng ⇒ show `3`. (3) `add(4)`, `add(3)` ⇒ còn `4` ⇒ show `4`. Kết quả `[None,2,None,3,None,None,4]`.

**Bài 2.** (1) `cnt = {5:2, 6:2, 7:1}`, `q = [5,6,5,7,6]`; show pop `5, 6, 5` ⇒ `q = [7,6]`, trả `7`. (2) `add(7)` ⇒ `cnt[7] = 2`, không append; show pop `7`, `6` ⇒ rỗng ⇒ `-1`. (3) `add(8)` ⇒ `q = [8]`; show ⇒ `8`.

**Bài 3.** (1) `c = self.cnt.get(value, 0) + 1`. (2) `c == 1` ⇒ `self.unique[value] = None`; `c == 2` ⇒ `del self.unique[value]`. (3) `show`: `next(iter(self.unique), -1)`.

**Bài 4.** (1) Với mỗi ký tự: `cnt[ch] += 1`, nếu `== 1` thì `q.append(ch)`. (2) `while q and cnt[q[0]] > 1: q.popleft()`. (3) Nối `q[0]` hoặc `'#'`.

**Bài 5.** (1) Như `FirstUnique`. (2) `showLastUnique`: `next(reversed(self.unique))` (OrderedDict hỗ trợ `reversed`). (3) `countUnique`: `len(self.unique)`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
ops1 = ["FirstUnique","showFirstUnique","add","showFirstUnique","add","add","showFirstUnique"]
args1 = [[[1,1,2,3]],[],[2],[],[4],[3],[]]
assert run_ops(FirstUnique, ops1, args1) == [None, 2, None, 3, None, None, 4]

# Bài 2
fu = FirstUniqueLazy([5, 6, 5, 7, 6])
assert fu.showFirstUnique() == 7 and list(fu.q) == [7, 6]
fu.add(7); assert fu.showFirstUnique() == -1 and list(fu.q) == []
fu.add(8); assert fu.showFirstUnique() == 8

# Bài 3
class FirstUniqueDictSol:
    def __init__(self, nums: list[int]):
        self.cnt = {}
        self.unique = {}
        for v in nums:
            self.add(v)
    def showFirstUnique(self) -> int:
        return next(iter(self.unique), -1)
    def add(self, value: int) -> None:
        c = self.cnt.get(value, 0) + 1
        self.cnt[value] = c
        if c == 1:
            self.unique[value] = None
        elif c == 2:
            del self.unique[value]

# Bài 4
def first_non_repeating_stream_sol(s):
    cnt, q, out = Counter(), deque(), []
    for ch in s:
        cnt[ch] += 1
        if cnt[ch] == 1:
            q.append(ch)
        while q and cnt[q[0]] > 1:
            q.popleft()
        out.append(q[0] if q else "#")
    return "".join(out)

assert first_non_repeating_stream_sol("aabc") == "a#bb"
assert first_non_repeating_stream_sol("zz") == "z#"

# Bài 5
class FirstLastUniqueSol:
    def __init__(self, nums):
        self.cnt = Counter(nums)
        self.unique = OrderedDict((v, None) for v in nums if self.cnt[v] == 1)
    def showFirstUnique(self):
        return next(iter(self.unique)) if self.unique else -1
    def showLastUnique(self):
        return next(reversed(self.unique)) if self.unique else -1
    def countUnique(self):
        return len(self.unique)
    def add(self, value):
        self.cnt[value] += 1
        if self.cnt[value] == 1:
            self.unique[value] = None
        elif value in self.unique:
            self.unique.pop(value)

rng = random.Random(9)
for _ in range(300):
    ops, args = random_ops(rng, rng.randint(1, 30), rng.choice([3, 8]))
    assert run_ops(FirstUniqueDictSol, ops, args) == run_ops(FirstUniqueBrute, ops, args)
    nums = args[0][0]
    a, queue = FirstLastUniqueSol(nums), list(nums)
    for op, ar in zip(ops[1:], args[1:]):
        if op == "add":
            a.add(ar[0]); queue.append(ar[0])
        c = Counter(queue)
        uniq = [x for x in queue if c[x] == 1]
        assert a.showFirstUnique() == (uniq[0] if uniq else -1)
        assert a.showLastUnique() == (uniq[-1] if uniq else -1)
        assert a.countUnique() == len(uniq)
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    exp = []
    for i in range(len(s)):
        c = Counter(s[:i + 1])
        exp.append(next((ch for ch in s[:i + 1] if c[ch] == 1), "#"))
    assert first_non_repeating_stream_sol(s) == "".join(exp)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phần tử duy nhất đầu tiên" trong luồng:**

```python
from collections import Counter, OrderedDict

class FirstUnique:
    def __init__(self, nums):
        self.cnt = Counter(nums)
        self.unique = OrderedDict((v, 1) for v in nums if self.cnt[v] == 1)
    def showFirstUnique(self):
        return next(iter(self.unique)) if self.unique else -1
    def add(self, value):
        self.cnt[value] += 1
        if self.cnt[value] == 1: self.unique[value] = 1
        elif value in self.unique: self.unique.pop(value)
```

**Template xoá lười:** `while q and bad(q[0]): q.popleft()` rồi đọc `q[0]`.

**Checklist:**
- [ ] Đếm cả các số trong `nums` ban đầu.
- [ ] Cấu trúc ứng viên **giữ thứ tự chèn** (không dùng `set`).
- [ ] Có bộ đếm — số trùng không "sống lại" ở lần thứ 3.
- [ ] Xoá lười dùng `while`, không phải `if`.
- [ ] `showFirstUnique` trả `-1` khi rỗng; `add` trả `None`.
- [ ] Đã test bằng bộ chạy chuỗi lệnh giả lập với ví dụ đề và chuỗi ngẫu nhiên.